# CIFAR-2 / ResNet-9 — LDS Benchmark (Colab GPU)

Compares **Traceprop** vs **TRAK (official, multi-checkpoint)** vs **Random baseline**.

**Runtime:** Set Runtime > Change runtime type > **T4 GPU**. Expected: ~1-2 hours.

TRAK uses the official `traker` library with 5 checkpoints (epochs 16-20), matching the methodology from Park et al. (2023).

Results saved to `cifar2_resnet9_lds.json` — download from Files panel.

In [1]:
!pip install -q traker[fast]

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU! Set Runtime > Change runtime type > T4 GPU")

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
PyTorch 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
import torchvision
import torchvision.transforms as transforms
from scipy.stats import spearmanr
import json
import time
import os

DEVICE = torch.device("cuda")
CKPT_DIR = "./checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# --- Config ---
PROJ_DIM = 4096
N_SUBSETS = 500
SUBSET_RATIO = 0.5
LR = 0.1
EPOCHS = 20
BATCH_SIZE = 128
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

def ckpt_exists(name):
    return os.path.exists(f"{CKPT_DIR}/{name}")

def save_ckpt(name, **kwargs):
    torch.save(kwargs, f"{CKPT_DIR}/{name}")
    print(f"  [Checkpoint saved: {name}]")

def load_ckpt(name):
    data = torch.load(f"{CKPT_DIR}/{name}", weights_only=False)
    print(f"  [Checkpoint loaded: {name}]")
    return data

## 1. CIFAR-2 Dataset (airplane vs automobile)

In [3]:
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

full_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_test)
full_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_indices = [i for i in range(len(full_train)) if full_train.targets[i] in (0, 1)]
test_indices = [i for i in range(len(full_test)) if full_test.targets[i] in (0, 1)]

train_subset = Subset(full_train, train_indices)
test_subset = Subset(full_test, test_indices)

N_TRAIN = len(train_indices)
N_TEST = len(test_indices)
print(f"CIFAR-2: {N_TRAIN} train, {N_TEST} test")

def preload(subset):
    xs, ys = [], []
    for x, y in subset:
        xs.append(x)
        ys.append(y)
    return torch.stack(xs), torch.tensor(ys)

X_train_all, y_train_all = preload(train_subset)
X_test_all, y_test_all = preload(test_subset)
print(f"X_train: {X_train_all.shape}, X_test: {X_test_all.shape}")

100%|██████████| 170M/170M [00:05<00:00, 29.5MB/s]


CIFAR-2: 10000 train, 2000 test
X_train: torch.Size([10000, 3, 32, 32]), X_test: torch.Size([2000, 3, 32, 32])


## 2. ResNet-9 Model

In [4]:
def conv_bn(in_c, out_c, kernel_size=3, stride=1, padding=1):
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, kernel_size, stride, padding, bias=False),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
    )

class ResNet9(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.prep = conv_bn(3, 64)
        self.layer1 = nn.Sequential(conv_bn(64, 128), nn.MaxPool2d(2))
        self.res1 = nn.Sequential(conv_bn(128, 128), conv_bn(128, 128))
        self.layer2 = nn.Sequential(conv_bn(128, 256), nn.MaxPool2d(2))
        self.layer3 = nn.Sequential(conv_bn(256, 512), nn.MaxPool2d(2))
        self.res2 = nn.Sequential(conv_bn(512, 512), conv_bn(512, 512))
        self.pool = nn.AdaptiveMaxPool2d(1)
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.prep(x)
        x = self.layer1(x)
        x = x + self.res1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = x + self.res2(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)

# Checkpoint epochs for TRAK multi-checkpoint (last 5 epochs)
CKPT_EPOCHS = [15, 16, 17, 18, 19]  # 0-indexed, so epochs 16-20

def train_resnet(model, X, y, lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, save_ckpts=None):
    """Train ResNet-9. If save_ckpts is a list, save state_dict at those epoch indices."""
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr,
        steps_per_epoch=(len(X) // batch_size + 1),
        epochs=epochs
    )
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)
    checkpoints = {}
    for epoch in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            scheduler.step()
        if save_ckpts is not None and epoch in save_ckpts:
            checkpoints[epoch] = {k: v.clone() for k, v in model.state_dict().items()}
    return model, checkpoints

@torch.no_grad()
def get_accuracy(model, X, y, batch_size=512):
    model.eval()
    correct = 0
    for xb, yb in DataLoader(TensorDataset(X, y), batch_size=batch_size):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        correct += (model(xb).argmax(1) == yb).sum().item()
    return correct / len(X)

@torch.no_grad()
def get_per_sample_correct(model, X, y, batch_size=512):
    """Return binary correctness per test sample."""
    model.eval()
    results = []
    for xb, yb in DataLoader(TensorDataset(X, y), batch_size=batch_size):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        results.append((model(xb).argmax(1) == yb).cpu().numpy())
    return np.concatenate(results).astype(np.float64)

@torch.no_grad()
def get_per_sample_margin(model, X, y, batch_size=512):
    """Return output margin (logit_correct - logit_incorrect) per test sample.
    This is what the TRAK paper uses for LDS — continuous, more informative than binary."""
    model.eval()
    margins = []
    for xb, yb in DataLoader(TensorDataset(X, y), batch_size=batch_size):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)  # (batch, 2)
        # For binary classification: margin = logit[correct] - logit[incorrect]
        correct_logit = logits.gather(1, yb.unsqueeze(1)).squeeze(1)
        # Get the other logit
        wrong_logit = logits.sum(1) - correct_logit  # works for 2 classes
        margin = (correct_logit - wrong_logit).cpu().numpy()
        margins.append(margin)
    return np.concatenate(margins).astype(np.float64)

n_params = sum(p.numel() for p in ResNet9().parameters())
print(f"ResNet-9 parameters: {n_params:,}")

ResNet-9 parameters: 6,569,026


## 3. Train Full Model

In [5]:
if ckpt_exists("model_multi.pt"):
    ckpt = load_ckpt("model_multi.pt")
    model = ResNet9(num_classes=2).to(DEVICE)
    model.load_state_dict(ckpt["final_state_dict"])
    full_acc = ckpt["full_acc"]
    train_time = ckpt["train_time"]
    multi_ckpts = ckpt["multi_ckpts"]  # dict: epoch -> state_dict
    print(f"Loaded {len(multi_ckpts)} checkpoints from epochs {sorted(multi_ckpts.keys())}")
else:
    torch.manual_seed(SEED)
    model = ResNet9(num_classes=2).to(DEVICE)
    t0 = time.perf_counter()
    model, multi_ckpts = train_resnet(model, X_train_all, y_train_all, save_ckpts=CKPT_EPOCHS)
    train_time = time.perf_counter() - t0
    full_acc = get_accuracy(model, X_test_all, y_test_all)
    save_ckpt("model_multi.pt",
              final_state_dict=model.state_dict(),
              multi_ckpts=multi_ckpts,
              full_acc=full_acc, train_time=train_time)
    print(f"Saved {len(multi_ckpts)} checkpoints from epochs {sorted(multi_ckpts.keys())}")

print(f"Full model accuracy: {full_acc:.4f} (trained in {train_time:.1f}s)")
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_params:,}")

  [Checkpoint saved: model_multi.pt]
Saved 5 checkpoints from epochs [15, 16, 17, 18, 19]
Full model accuracy: 0.9725 (trained in 79.7s)
Trainable params: 6,569,026


## 4. TRAK Attribution (official library, multi-checkpoint)

In [6]:
if ckpt_exists("trak_influence.pt"):
    ckpt = load_ckpt("trak_influence.pt")
    trak_influence_matrix = ckpt["trak_influence_matrix"]
    trak_time = ckpt["trak_time"]
    print(f"TRAK loaded: {trak_influence_matrix.shape}, took {trak_time:.1f}s originally")
else:
    from trak import TRAKer

    print(f"Computing TRAK attribution with {len(multi_ckpts)} checkpoints...")
    t0 = time.perf_counter()

    traker = TRAKer(model=ResNet9(num_classes=2).to(DEVICE),
                    task='image_classification',
                    train_set_size=N_TRAIN,
                    proj_dim=PROJ_DIM,
                    save_dir='./trak_cache',
                    device=DEVICE,
                    proj_max_batch_size=8,
                    use_half_precision=False)

    # Featurize training data for each checkpoint
    loader_train = DataLoader(TensorDataset(X_train_all, y_train_all),
                              batch_size=16, shuffle=False)
    for model_id, epoch in enumerate(sorted(multi_ckpts.keys())):
        sd = multi_ckpts[epoch]
        traker.load_checkpoint(sd, model_id=model_id)
        for xb, yb in loader_train:
            traker.featurize(batch=(xb.to(DEVICE), yb.to(DEVICE)),
                           num_samples=xb.shape[0])
        print(f"  Featurized checkpoint {model_id} (epoch {epoch+1})")
    traker.finalize_features()
    print("  Features finalized")

    # Score test data for each checkpoint
    loader_test = DataLoader(TensorDataset(X_test_all, y_test_all),
                             batch_size=16, shuffle=False)
    for model_id, epoch in enumerate(sorted(multi_ckpts.keys())):
        sd = multi_ckpts[epoch]
        traker.start_scoring_checkpoint(exp_name='lds',
                                        checkpoint=sd,
                                        model_id=model_id,
                                        num_targets=N_TEST)
        for xb, yb in loader_test:
            traker.score(batch=(xb.to(DEVICE), yb.to(DEVICE)),
                        num_samples=xb.shape[0])
        print(f"  Scored checkpoint {model_id} (epoch {epoch+1})")

    trak_scores = traker.finalize_scores(exp_name='lds')
    trak_influence_matrix = np.array(trak_scores)  # shape: (N_TEST, N_TRAIN)
    trak_time = time.perf_counter() - t0
    save_ckpt("trak_influence.pt", trak_influence_matrix=trak_influence_matrix, trak_time=trak_time)
    print(f"TRAK done: {trak_influence_matrix.shape}, {trak_time:.1f}s")

INFO:STORE:No existing model IDs in /content/trak_cache.
INFO:STORE:No existing TRAK scores in /content/trak_cache.


Computing TRAK attribution with 5 checkpoints...
  Featurized checkpoint 0 (epoch 16)
  Featurized checkpoint 1 (epoch 17)
  Featurized checkpoint 2 (epoch 18)
  Featurized checkpoint 3 (epoch 19)
  Featurized checkpoint 4 (epoch 20)


Finalizing features for all model IDs..: 100%|██████████| 5/5 [00:05<00:00,  1.10s/it]


  Features finalized
  Scored checkpoint 0 (epoch 16)
  Scored checkpoint 1 (epoch 17)
  Scored checkpoint 2 (epoch 18)
  Scored checkpoint 3 (epoch 19)
  Scored checkpoint 4 (epoch 20)


Finalizing scores for all model IDs..: 100%|██████████| 5/5 [00:00<00:00,  5.88it/s]
INFO:STORE:Saving scores in /content/trak_cache/scores/lds.mmap


  [Checkpoint saved: trak_influence.pt]
TRAK done: (10000, 2000), 659.0s


## 5. Traceprop Attribution (batch-mean gradients, JL projection)

Implements the Traceprop methodology:
- **Batch-mean gradients**: one gradient per batch, assigned to all samples (same as `TrainingContext.step()`)
- **Count-sketch JL projection**: O(n_params) memory — a valid JL embedding equivalent to the library's sparse JL
  - The library uses dense sparse-JL `{-1,0,+1}` matrix which requires `O(proj_dim × n_params)` memory
  - For 6.5M-param ResNet-9, that's ~100GB — impractical on Colab
  - Count-sketch provides the same JL guarantee with O(n_params) storage
- **Dot-product attribution** (no normalization, for LDS evaluation)
- Seed 42 (matching library default)

In [7]:
if ckpt_exists("tp_influence.pt"):
    ckpt = load_ckpt("tp_influence.pt")
    tp_influence_matrix = ckpt["tp_influence_matrix"]
    tp_time = ckpt["tp_time"]
    print(f"Traceprop loaded: {tp_influence_matrix.shape}, took {tp_time:.1f}s originally")
else:
    print("Computing Traceprop attribution (batch-mean, count-sketch JL)...")
    t0 = time.perf_counter()

    # Count-sketch projection (valid JL embedding, O(n_params) memory)
    torch.manual_seed(42)
    tp_hash_buckets = torch.randint(0, PROJ_DIM, (n_params,), device=DEVICE)
    tp_hash_signs = (torch.randint(0, 2, (n_params,), device=DEVICE).float() * 2 - 1) * (1.0 / PROJ_DIM) ** 0.5

    def tp_project(full_grad):
        proj = torch.zeros(PROJ_DIM, device=DEVICE)
        proj.scatter_add_(0, tp_hash_buckets, full_grad * tp_hash_signs)
        return proj

    model.eval()
    loss_fn = nn.CrossEntropyLoss()

    # Batch-mean gradients (matching TrainingContext.step behavior)
    loader = DataLoader(TensorDataset(X_train_all, y_train_all), batch_size=BATCH_SIZE, shuffle=False)
    batch_projs = []
    batch_sizes = []
    for batch_i, (xb, yb) in enumerate(loader):
        model.zero_grad()
        out = model(xb.to(DEVICE))
        loss = loss_fn(out, yb.to(DEVICE))
        loss.backward()
        full_grad = torch.cat([p.grad.flatten() for p in model.parameters() if p.requires_grad])
        proj = tp_project(full_grad)
        batch_projs.append(proj)
        batch_sizes.append(xb.shape[0])
        if batch_i % 20 == 0:
            print(f"  Train batch {batch_i}/{len(loader)} ({time.perf_counter()-t0:.0f}s)")

    # Expand batch-level to sample-level (each sample gets its batch gradient)
    tp_train_projs = torch.zeros(N_TRAIN, PROJ_DIM, device=DEVICE)
    idx = 0
    for proj, bs in zip(batch_projs, batch_sizes):
        tp_train_projs[idx:idx+bs] = proj.unsqueeze(0)
        idx += bs

    # Test gradients (per-sample)
    tp_test_projs = torch.zeros(N_TEST, PROJ_DIM, device=DEVICE)
    for i in range(N_TEST):
        model.zero_grad()
        out = model(X_test_all[i].unsqueeze(0).to(DEVICE))
        loss = loss_fn(out, y_test_all[i].unsqueeze(0).to(DEVICE))
        loss.backward()
        full_grad = torch.cat([p.grad.flatten() for p in model.parameters() if p.requires_grad])
        tp_test_projs[i] = tp_project(full_grad)
        if i % 500 == 0:
            print(f"  Test {i}/{N_TEST} ({time.perf_counter()-t0:.0f}s)")

    tp_influence_matrix = (tp_test_projs @ tp_train_projs.T).cpu().numpy()
    tp_time = time.perf_counter() - t0
    save_ckpt("tp_influence.pt", tp_influence_matrix=tp_influence_matrix, tp_time=tp_time)
    print(f"Traceprop done: {tp_influence_matrix.shape}, {tp_time:.1f}s")

Computing Traceprop attribution (batch-mean, count-sketch JL)...
  Train batch 0/79 (0s)
  Train batch 20/79 (1s)
  Train batch 40/79 (2s)
  Train batch 60/79 (3s)
  Test 0/2000 (4s)
  Test 500/2000 (7s)
  Test 1000/2000 (9s)
  Test 1500/2000 (11s)
  [Checkpoint saved: tp_influence.pt]
Traceprop done: (2000, 10000), 13.9s


## 6. Ground-Truth Retraining (50 subsets)

In [8]:
print(f"Retraining {N_SUBSETS} models for ground truth LDS...")
t0 = time.perf_counter()

np.random.seed(SEED + 1000)
all_masks = [np.random.rand(N_TRAIN) < SUBSET_RATIO for _ in range(N_SUBSETS)]

# Resume from checkpoint if available
if ckpt_exists("retrain_margin.pt"):
    ckpt = load_ckpt("retrain_margin.pt")
    subset_masks = ckpt["subset_masks"]
    subset_margins = ckpt["subset_margins"]
    start_s = int(ckpt["completed"])
    print(f"  Resuming from subset {start_s}/{N_SUBSETS}")
else:
    subset_masks = np.array(all_masks)
    subset_margins = np.zeros((N_SUBSETS, N_TEST))
    start_s = 0

for s in range(start_s, N_SUBSETS):
    mask = all_masks[s]
    torch.manual_seed(s)
    m = ResNet9(num_classes=2).to(DEVICE)
    m, _ = train_resnet(m, X_train_all[mask], y_train_all[mask])
    subset_margins[s] = get_per_sample_margin(m, X_test_all, y_test_all)

    acc = (subset_margins[s] > 0).mean()
    elapsed = time.perf_counter() - t0
    done = s - start_s + 1
    eta = elapsed / done * (N_SUBSETS - s - 1)
    print(f"  Subset {s+1}/{N_SUBSETS}: acc={acc:.4f} [{elapsed/60:.0f}m elapsed, ~{eta/60:.0f}m remaining]")

    save_ckpt("retrain_margin.pt",
              subset_masks=subset_masks, subset_margins=subset_margins, completed=s+1)

retrain_time = time.perf_counter() - t0
print(f"\nRetraining took {retrain_time/60:.1f} min")

Retraining 500 models for ground truth LDS...
  Subset 1/500: acc=0.9230  elapsed, ~341m remaining]
  [Checkpoint saved: retrain_margin.pt]
  Subset 2/500: acc=0.9115  elapsed, ~341m remaining]
  [Checkpoint saved: retrain_margin.pt]
  Subset 3/500: acc=0.9345  elapsed, ~340m remaining]
  [Checkpoint saved: retrain_margin.pt]
  Subset 4/500: acc=0.9395  elapsed, ~340m remaining]
  [Checkpoint saved: retrain_margin.pt]
  Subset 5/500: acc=0.9295  elapsed, ~340m remaining]
  [Checkpoint saved: retrain_margin.pt]
  Subset 6/500: acc=0.9290  elapsed, ~339m remaining]
  [Checkpoint saved: retrain_margin.pt]
  Subset 7/500: acc=0.9065  elapsed, ~338m remaining]
  [Checkpoint saved: retrain_margin.pt]
  Subset 8/500: acc=0.9305  elapsed, ~337m remaining]
  [Checkpoint saved: retrain_margin.pt]
  Subset 9/500: acc=0.9160  elapsed, ~337m remaining]
  [Checkpoint saved: retrain_margin.pt]
  Subset 10/500: acc=0.9425  elapsed, ~336m remaining]
  [Checkpoint saved: retrain_margin.pt]
  Subset 11/5

## 7. Compute LDS

In [9]:
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Fix TRAK matrix orientation if needed
if trak_influence_matrix.shape == (N_TRAIN, N_TEST):
    trak_influence_matrix = trak_influence_matrix.T
    print(f"Transposed TRAK matrix to {trak_influence_matrix.shape}")

def compute_lds(influence_matrix, subset_masks, subset_outcomes, n_test):
    """
    Correct LDS (Linear Datamodeling Score) per Park et al. 2023:
    For each test point, predict subset outcomes as linear combination
    of attribution scores, then Spearman correlate with actual outcomes (margins).
    """
    lds_scores = []
    for test_i in range(n_test):
        predicted = subset_masks @ influence_matrix[test_i]  # (n_subsets,)
        actual = subset_outcomes[:, test_i]  # (n_subsets,)
        corr = spearmanr(predicted, actual).statistic
        lds_scores.append(corr if not np.isnan(corr) else 0.0)
        if test_i % 200 == 0:
            print(f"  Test point {test_i}/{n_test}")
    return np.array(lds_scores)

print("Computing LDS (margin-based)...")

print("\nTRAK:")
lds_trak = compute_lds(trak_influence_matrix, subset_masks, subset_margins, N_TEST)
print(f"  TRAK LDS: {lds_trak.mean():.4f} +/- {lds_trak.std():.4f}")

print("\nTraceprop:")
lds_tp = compute_lds(tp_influence_matrix, subset_masks, subset_margins, N_TEST)
print(f"  Traceprop LDS: {lds_tp.mean():.4f} +/- {lds_tp.std():.4f}")

print("\nRandom:")
np.random.seed(SEED + 2000)
random_matrix = np.random.rand(N_TEST, N_TRAIN)
lds_rand = compute_lds(random_matrix, subset_masks, subset_margins, N_TEST)
print(f"  Random LDS: {lds_rand.mean():.4f} +/- {lds_rand.std():.4f}")

Transposed TRAK matrix to (2000, 10000)
Computing LDS (margin-based)...

TRAK:
  Test point 0/2000
  Test point 200/2000
  Test point 400/2000
  Test point 600/2000
  Test point 800/2000
  Test point 1000/2000
  Test point 1200/2000
  Test point 1400/2000
  Test point 1600/2000
  Test point 1800/2000
  TRAK LDS: 0.0323 +/- 0.0507

Traceprop:
  Test point 0/2000
  Test point 200/2000
  Test point 400/2000
  Test point 600/2000
  Test point 800/2000
  Test point 1000/2000
  Test point 1200/2000
  Test point 1400/2000
  Test point 1600/2000
  Test point 1800/2000
  Traceprop LDS: 0.0147 +/- 0.0342

Random:
  Test point 0/2000
  Test point 200/2000
  Test point 400/2000
  Test point 600/2000
  Test point 800/2000
  Test point 1000/2000
  Test point 1200/2000
  Test point 1400/2000
  Test point 1600/2000
  Test point 1800/2000
  Random LDS: -0.0018 +/- 0.0350


In [10]:
print("\n" + "=" * 60)
print("CIFAR-2 / ResNet-9 — LDS Results")
print("=" * 60)
print(f"{'Method':<30} {'LDS (mean +/- std)':>20}")
print("-" * 55)
print(f"{'TRAK (5 ckpts, official)':<30} {lds_trak.mean():>8.4f} +/- {lds_trak.std():.4f}")
print(f"{'Traceprop (batch-mean)':<30} {lds_tp.mean():>8.4f} +/- {lds_tp.std():.4f}")
print(f"{'Random baseline':<30} {lds_rand.mean():>8.4f} +/- {lds_rand.std():.4f}")
print("=" * 60)

results = {
    "benchmark": "CIFAR-2/ResNet-9",
    "trak_lds_mean": round(float(lds_trak.mean()), 4),
    "trak_lds_std": round(float(lds_trak.std()), 4),
    "traceprop_lds_mean": round(float(lds_tp.mean()), 4),
    "traceprop_lds_std": round(float(lds_tp.std()), 4),
    "random_lds_mean": round(float(lds_rand.mean()), 4),
    "random_lds_std": round(float(lds_rand.std()), 4),
    "full_model_accuracy": round(full_acc, 4),
    "n_train": N_TRAIN,
    "n_test": N_TEST,
    "n_subsets": N_SUBSETS,
    "n_trak_checkpoints": len(CKPT_EPOCHS),
    "proj_dim": PROJ_DIM,
    "epochs": EPOCHS,
    "trak_time_s": round(trak_time, 1),
    "traceprop_time_s": round(tp_time, 1),
    "retrain_time_s": round(retrain_time, 1),
    "trak_method": "official_traker_multi_checkpoint",
    "device": str(DEVICE),
}

with open("cifar2_resnet9_lds.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved to cifar2_resnet9_lds.json")
print(json.dumps(results, indent=2))


CIFAR-2 / ResNet-9 — LDS Results
Method                           LDS (mean +/- std)
-------------------------------------------------------
TRAK (5 ckpts, official)         0.0323 +/- 0.0507
Traceprop (batch-mean)           0.0147 +/- 0.0342
Random baseline                 -0.0018 +/- 0.0350

Saved to cifar2_resnet9_lds.json
{
  "benchmark": "CIFAR-2/ResNet-9",
  "trak_lds_mean": 0.0323,
  "trak_lds_std": 0.0507,
  "traceprop_lds_mean": 0.0147,
  "traceprop_lds_std": 0.0342,
  "random_lds_mean": -0.0018,
  "random_lds_std": 0.035,
  "full_model_accuracy": 0.9725,
  "n_train": 10000,
  "n_test": 2000,
  "n_subsets": 500,
  "n_trak_checkpoints": 5,
  "proj_dim": 4096,
  "epochs": 20,
  "trak_time_s": 659.0,
  "traceprop_time_s": 13.9,
  "retrain_time_s": 20599.5,
  "trak_method": "official_traker_multi_checkpoint",
  "device": "cuda"
}
